In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc qiskit-addon-aqc-tensor qiskit-addon-utils scipy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Introducere în compilarea cuantică aproximativă cu rețele tensoare (AQC-Tensor)


<details>
<summary><b>Versiuni de pachete</b></summary>

Codul de pe această pagină a fost dezvoltat folosind următoarele cerințe.
Recomandăm folosirea acestor versiuni sau a unora mai noi.

```
qiskit[all]~=2.3.0
qiskit-aer~=0.17
qiskit-addon-utils~=0.3.0
qiskit-addon-aqc-tensor[aer,quimb-jax]~=0.2.0; sys.platform != 'darwin'
scipy~=1.16.3
```
</details>
Acest ghid demonstrează un exemplu simplu de funcționare pentru a începe cu AQC-Tensor. În acest exemplu, vei prelua un Circuit Trotter care simulează evoluția unui model Ising în câmp transversal și vei folosi metoda AQC-Tensor pentru a reduce adâncimea circuitului rezultat. În plus, acest exemplu necesită pachetul `qiskit-addon-utils` pentru generatorul de probleme, `qiskit-aer` pentru simularea rețelei tensoare și `scipy` pentru optimizarea parametrilor.

Pentru a începe, reamintim că Hamiltonianul modelului Ising în câmp transversal are forma

$$ \mathcal{H}_{Ising} = \sum_{i=1}^N J_{i,(i+1)}Z_iZ_{i+1} + h_i X_i $$

unde vom presupune condiții periodice la frontieră, ceea ce implică faptul că pentru $i=10$ obținem $i+1=11\rightarrow 1$, iar $J$ este intensitatea cuplajului dintre două site-uri și $h$ este intensitatea câmpului magnetic extern.

Fragmentul de cod următor va genera Hamiltonianul unui lanț Ising cu 10 site-uri și condiții periodice la frontieră.

In [1]:
from qiskit.transpiler import CouplingMap
from qiskit_addon_utils.problem_generators import generate_xyz_hamiltonian
from qiskit.synthesis import SuzukiTrotter
from qiskit_addon_utils.problem_generators import (
    generate_time_evolution_circuit,
)
from qiskit_addon_aqc_tensor import generate_ansatz_from_circuit
from qiskit_addon_aqc_tensor.simulation import tensornetwork_from_circuit
from qiskit_addon_aqc_tensor.simulation import compute_overlap
from qiskit_addon_aqc_tensor.objective import MaximizeStateFidelity
from qiskit_aer import AerSimulator
from scipy.optimize import OptimizeResult, minimize


# Generate some coupling map to use for this example
coupling_map = CouplingMap.from_heavy_hex(3, bidirectional=False)

# Choose a 10-qubit circle on this coupling map
reduced_coupling_map = coupling_map.reduce(
    [0, 13, 1, 14, 10, 16, 4, 15, 3, 9]
)

# Get a qubit operator describing the Ising field model
hamiltonian = generate_xyz_hamiltonian(
    reduced_coupling_map,
    coupling_constants=(0.0, 0.0, 1.0),
    ext_magnetic_field=(0.4, 0.0, 0.0),
)

## Împărțirea evoluției temporale în două părți pentru execuția clasică și cuantică
Scopul general al acestui exemplu este de a simula evoluția temporală a Hamiltonianului modelului. Facem acest lucru prin evoluție Trotter, care va fi împărțită în două porțiuni.

1. O porțiune inițială care poate fi simulată folosind stări produs matriceal (MPS). Aceasta este cea care va fi „compilată" folosind AQC-Tensor.
2. O porțiune ulterioară care va fi executată pe hardware cuantic.

Vom alege să evoluăm sistemul până la timpul $t_f=5$ și vom folosi AQC-Tensor pentru a comprima evoluția temporală până la timpul $t=4$, apoi vom evolua folosind pași Trotter obișnuiți până la $t_f$.

De aici, vom genera în continuare două Circuit-uri: unul care va fi comprimat folosind AQC-Tensor și unul care va fi executat pe un QPU. Pentru primul Circuit, deoarece acesta va fi simulat clasic folosind stări produs matriceal, vom utiliza un număr generos de straturi pentru a minimiza eroarea Trotter. Între timp, al doilea Circuit care simulează evoluția temporală de la $t_i=4$ la $t_f=5$ va folosi mult mai puține straturi pentru a minimiza adâncimea.

In [2]:
# Generate circuit to be compressed
aqc_evolution_time = 4.0
aqc_target_num_trotter_steps = 45

aqc_target_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_target_num_trotter_steps),
    time=aqc_evolution_time,
)

# Generate circuit to execute on hardware
subsequent_evolution_time = 1.0
subsequent_num_trotter_steps = 5

subsequent_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=subsequent_num_trotter_steps),
    time=subsequent_evolution_time,
)

În scopuri comparative, vom genera și un al treilea Circuit: unul care evoluează până la $t=4$, dar care are același număr de straturi ca și al doilea Circuit care evoluează de la $t_i=4$ la $t_f=5$. Acesta este Circuit-ul pe care *l-am fi* executat dacă nu am fi folosit tehnica AQC-Tensor. Îl vom numi Circuit-ul de *comparație* pentru moment.

In [3]:
aqc_comparison_num_trotter_steps = int(
    subsequent_num_trotter_steps
    / subsequent_evolution_time
    * aqc_evolution_time
)
aqc_comparison_num_trotter_steps

comparison_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_comparison_num_trotter_steps),
    time=aqc_evolution_time,
)

## Generarea ansatz-ului și construirea simulației MPS
Ulterior, vom genera ansatz-ul pe care îl vom optimiza. Acesta va evolua până la același timp de evoluție ca și primul nostru Circuit (de la $t_i=0$ la $t_f=4$), dar cu mai puțini pași Trotter.

Odată ce Circuit-ul a fost generat, îl transmitem funcției `generate_ansatz_from_circuit()` din AQC-Tensor, care analizează conectivitatea pe doi Qubiți și returnează două lucruri. Primul este un Circuit ansatz generat cu aceeași conectivitate pe doi Qubiți, iar al doilea este un set de parametri care, introduși în ansatz, produc Circuit-ul de intrare.

In [4]:
aqc_ansatz_num_trotter_steps = 5

aqc_good_circuit = generate_time_evolution_circuit(
    hamiltonian,
    synthesis=SuzukiTrotter(reps=aqc_ansatz_num_trotter_steps),
    time=aqc_evolution_time,
)

aqc_ansatz, aqc_initial_parameters = generate_ansatz_from_circuit(
    aqc_good_circuit, qubits_initially_zero=True
)
aqc_ansatz.draw("mpl", fold=-1)

<Image src="../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/e08edb92-da1f-4131-85f5-f89006f7a2dd-0.svg" alt="Output of the previous code cell" />

![Output of the previous code cell](../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/e08edb92-da1f-4131-85f5-f89006f7a2dd-0.svg)

Ulterior, vom construi reprezentarea MPS a stării care urmează să fie aproximată de AQC. Vom calcula de asemenea fidelitatea $|\langle\psi_1 | \psi_2 \rangle |^2$ dintre starea pregătită de Circuit-ul de comparație față de Circuit-ul care generează starea țintă (care a folosit mai mulți pași Trotter).

In [5]:
# Generate MPS simulator settings and obtain the MPS representation of the target state
simulator_settings = AerSimulator(
    method="matrix_product_state",
    matrix_product_state_max_bond_dimension=100,
)
aqc_target_mps = tensornetwork_from_circuit(
    aqc_target_circuit, simulator_settings
)


# Compute the fidelity between the MPS representation of the target state and the state produced by the comparison circuit
comparison_mps = tensornetwork_from_circuit(
    comparison_circuit, simulator_settings
)
comparison_fidelity = (
    abs(compute_overlap(comparison_mps, aqc_target_mps)) ** 2
)
print(f"Comparison fidelity: {comparison_fidelity}")

Comparison fidelity: 0.9997111919739693


## Optimize the parameters of the ansatz using the MPS

Lastly, we will optimize the ansatz circuit such that it produces the target state better than our `comparison_fidelity`. The cost function to minimize will be the `MaximizeStateFidelity` and will be optimized using scipy's L-BFGS optimizer.

In [6]:
objective = MaximizeStateFidelity(
    aqc_target_mps, aqc_ansatz, simulator_settings
)

stopping_point = 1 - comparison_fidelity


def callback(intermediate_result: OptimizeResult):
    print(f"Intermediate result: Fidelity {1 - intermediate_result.fun:.8}")
    if intermediate_result.fun < stopping_point:
        # Good enough for now
        raise StopIteration


result = minimize(
    objective,
    aqc_initial_parameters,
    method="L-BFGS-B",
    jac=True,
    options={"maxiter": 100},
    callback=callback,
)
if (
    result.status
    not in (
        0,
        1,
        99,
    )
):  # 0 => success; 1 => max iterations reached; 99 => early termination via StopIteration
    raise RuntimeError(
        f"Optimization failed: {result.message} (status={result.status})"
    )

print(f"Done after {result.nit} iterations.")
aqc_final_parameters = result.x

Intermediate result: Fidelity 0.95084365


Intermediate result: Fidelity 0.98409893


Intermediate result: Fidelity 0.99142033


Intermediate result: Fidelity 0.99521405


Intermediate result: Fidelity 0.99566673


...

Intermediate result: Fidelity 0.99956469


Intermediate result: Fidelity 0.99964488


Intermediate result: Fidelity 0.99967419


Intermediate result: Fidelity 0.99968821


Intermediate result: Fidelity 0.9997448
Done after 24 iterations.


## Optimizarea parametrilor ansatz-ului folosind MPS
În final, vom optimiza Circuit-ul ansatz astfel încât acesta să producă starea țintă cu o fidelitate mai mare decât `comparison_fidelity`. Funcția de cost de minimizat va fi `MaximizeStateFidelity` și va fi optimizată folosind optimizatorul L-BFGS din scipy.

In [7]:
final_circuit = aqc_ansatz.assign_parameters(aqc_final_parameters)
final_circuit.compose(subsequent_circuit, inplace=True)
final_circuit.draw("mpl", fold=-1)

<Image src="../docs/images/guides/qiskit-addons-aqc-get-started/extracted-outputs/45abbabe-0289-4a09-aa99-89f70bdc535d-0.svg" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
    - Try the [Approximate quantum compilation for time evolution circuits](/docs/tutorials/approximate-quantum-compilation-for-time-evolution) tutorial.
</Admonition>